# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Kaggle notebook setup
### Installations

In [1]:
!pip install mlflow --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 72.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 63.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.2/789.2 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behav

### Data location

In [2]:
import os
print(os.listdir('/kaggle/input/'))
print(os.listdir('/kaggle/input/radimagenet-densenet121-notop'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed/processed'))

['brain-tumor-mri-preprocessed', 'radimagenet-densenet121-notop']
['RadImageNet-DenseNet121_notop.h5']
['processed']
['Validation', 'Training', 'Testing', '.gitkeep']


## General

In [12]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("Brain_Tumor_Training")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'preachingly-nonabjuratory-marget.ngrok-free.dev'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Experiment: artifact_location='mlflow-artifacts:/830660600119881173', creation_time=1769099937797, experiment_id='830660600119881173', last_update_time=1769099937797, lifecycle_stage='active', name='Brain_Tumor_Training', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import math

2026-01-28 14:42:02.794380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769611322.967712      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769611323.015757      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769611323.400173      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769611323.400221      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769611323.400224      55 computation_placer.cc:177] computation placer alr

In [5]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices('GPU')

Num GPUs Available: 1


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [30]:
# path management
PROJECT_ROOT = '/kaggle/input'
PREP_DIR = PROJECT_ROOT + '/brain-tumor-mri-preprocessed/processed'
ARTEFACTS_DIR = PROJECT_ROOT + '/radimagenet-densenet121-notop'

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

PROJECT_NAME = "BrainTumorMRI"
MODEL_TYPE = "DenseNet121"
TWO_HEAD = True
MODEL_NAME = f"{PROJECT_NAME}_{MODEL_TYPE}_{TWO_HEAD*"2Head"}"
FREEZE_BACKBONE = True
MASK_TUMOR_TYPE_LOSS = True
BATCH_SIZE = 32
DATA_AUGMENTATION = True

## Modeling

### Backbone

In [77]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = not FREEZE_BACKBONE

print("✅ RadImageNet DenseNet121 loaded successfully")

✅ RadImageNet DenseNet121 loaded successfully


In [78]:
#backbone.summary()

In [79]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [80]:
model_data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomZoom((-0.03,0.03),(-0.03,0.03), seed=SEED),
    layers.RandomTranslation((-0.01,0.01),(-0.01,0.01), seed=SEED),
], name='data_augmentation_part')

In [81]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [82]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [83]:
def shared_head_part(inputs, backbone, data_augmentation):
    # Data augmentation (training only)
    x = data_augmentation(inputs)
    # Backbone - force into inference
    x = backbone(x, training=False)

    # Shared head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    return x

In [84]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = shared_head_part(inputs, backbone, model_data_augmentation)

#Heads
output_presence = model_head1(x)
output_type = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output_presence,
        "tumor_type": output_type
    },
    name='densenet_two_head'
)

In [85]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection (penalize false negatives), but taking account that tumors are 75% of data
)

In [86]:
def masked_sparse_cce(y_true, y_pred):
    tumor_present = tf.cast(y_true != 0, tf.float32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss = loss * tumor_present
    return tf.reduce_sum(loss) / (tf.reduce_sum(tumor_present) + 1e-6)

In [87]:
loss_weight_presence = 1.0
loss_weight_type = 1.3 # we give a little more weight to the classification of the type

model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": masked_sparse_cce,
    },
    
    loss_weights={
        "tumor_presence": loss_weight_presence,
        "tumor_type": loss_weight_type, 
    },
    
    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": ["accuracy"],
    }
)

In [88]:
#model.summary()

## Streaming Training

In [89]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [90]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=BATCH_SIZE,\n    repeat=False\n).prefetch(tf.data.AUTOTUNE)\n'

In [91]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }


In [92]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [93]:
#for x, y in train_ds.take(1):
#    print("Image:")
#    print(x.dtype, x.shape)
#    print("\nLabels:")
#    for k, v in y.items():
#        print(k, v.dtype, v.shape)

In [94]:
def count_tfrecord_batches(directory_path, batch_size):
    """
    Count number of batches for a TFRecord dataset.

    Assumes:
    - 1 sample per TFRecord
    - batch_size is variable

    Returns:
    - number of batches = ceil(num_samples / batch_size)
    """
    directory_path = Path(directory_path)
    n_samples = len(list(directory_path.glob("*.tfrecord")))
    return math.ceil(n_samples / batch_size)


In [95]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})


In [96]:
#to_monitor = "val_tumor_presence_recall"
to_monitor = "val_tumor_type_loss"

reduce_lr = ReduceLROnPlateau(
    monitor=to_monitor,
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor=to_monitor,
    mode="max",
    min_delta=0.0001,
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [97]:
#print("RUN_NAME:", RUN_NAME)
#print("TRAIN steps:", count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE))
#print("VAL steps:", count_tfrecord_batches(VAL_DIR, BATCH_SIZE))
#print("train_ds element_spec:", train_ds.element_spec)

In [98]:
RUN_NAME = (
    f"{MODEL_TYPE}"
    f"freeze={FREEZE_BACKBONE}_"
    f"mask={MASK_TUMOR_TYPE_LOSS}_"
    f"{datetime.now().strftime('%Y%m%d-%H%M')}"
)

print(f"Run name: {RUN_NAME}\n")

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.tensorflow.autolog(registered_model_name=MODEL_NAME)

    mlflow.log_params({
        "project": PROJECT_NAME,
        "model_type": MODEL_TYPE,
        "pretrained_weights": "RadImageNet",
        "backbone_frozen": FREEZE_BACKBONE,
        "head_1": "tumor_presence_binary",
        "head_2": "tumor_type_softmax",
        "loss_weight_presence": loss_weight_presence,
        "loss_weight_type": loss_weight_type,
        "mask_tumor_type_loss": MASK_TUMOR_TYPE_LOSS,
        "label_0_means": "no_tumor",
        "data_augmentation": DATA_AUGMENTATION,
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        #steps_per_epoch=count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE),
        #validation_steps=count_tfrecord_batches(VAL_DIR, BATCH_SIZE),
        callbacks=[reduce_lr, early_stopping],
        verbose=1,
    )

    client = MlflowClient()
    latest_version = client.get_latest_versions(MODEL_NAME)[-1].version

    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging"
    )


Run name: DenseNet121freeze=True_mask=True_20260128-1606



2026/01/28 16:06:47 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.
2026/01/28 16:06:49 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.


Epoch 1/50
    143/Unknown 37s 143ms/step - loss: 1.3622 - tumor_presence_accuracy: 0.8004 - tumor_presence_auc: 0.8275 - tumor_presence_loss: 0.1313 - tumor_presence_precision: 0.8567 - tumor_presence_recall: 0.8700 - tumor_type_accuracy: 0.5222 - tumor_type_loss: 0.9468

143/143 ━━━━━━━━━━━━━━━━━━━━ 61s 310ms/step - loss: 1.3599 - tumor_presence_accuracy: 0.8007 - tumor_presence_auc: 0.8279 - tumor_presence_loss: 0.1311 - tumor_presence_precision: 0.8569 - tumor_presence_recall: 0.8702 - tumor_type_accuracy: 0.5224 - tumor_type_loss: 0.9452 - val_loss: 2.7243 - val_tumor_presence_accuracy: 0.2878 - val_tumor_presence_auc: 0.9655 - val_tumor_presence_loss: 1.0210 - val_tumor_presence_precision: 1.0000 - val_tumor_presence_recall: 0.0121 - val_tumor_type_accuracy: 0.4383 - val_tumor_type_loss: 1.2528 - learning_rate: 0.0010
Epoch 2/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 174ms/step - loss: 0.7641 - tumor_presence_accuracy: 0.8999 - tumor_presence_auc: 0.9450 - tumor_presence_loss: 0.0744 - tumor_presence_precision: 0.9284 - tumor_presence_recall: 0.9336 - tumor_type_accuracy: 0.5719 - tumor_type_loss: 0.5305 - val_loss: 3.9223 - val_tumor_presence_accuracy: 0.7340 - val_tumor_presence_auc: 0.9402 - val_tumor_presence_loss: 0.1478 - val_tumor_presence_precisi

143/143 ━━━━━━━━━━━━━━━━━━━━ 44s 307ms/step - loss: 0.6209 - tumor_presence_accuracy: 0.9197 - tumor_presence_auc: 0.9610 - tumor_presence_loss: 0.0604 - tumor_presence_precision: 0.9373 - tumor_presence_recall: 0.9522 - tumor_type_accuracy: 0.5897 - tumor_type_loss: 0.4311 - val_loss: 2.6476 - val_tumor_presence_accuracy: 0.8303 - val_tumor_presence_auc: 0.9394 - val_tumor_presence_loss: 0.1143 - val_tumor_presence_precision: 0.9646 - val_tumor_presence_recall: 0.7937 - val_tumor_type_accuracy: 0.2511 - val_tumor_type_loss: 1.8964 - learning_rate: 0.0010
Epoch 5/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 174ms/step - loss: 0.5665 - tumor_presence_accuracy: 0.9270 - tumor_presence_auc: 0.9723 - tumor_presence_loss: 0.0491 - tumor_presence_precision: 0.9479 - tumor_presence_recall: 0.9518 - tumor_type_accuracy: 0.6044 - tumor_type_loss: 0.3980 - val_loss: 4.3008 - val_tumor_presence_accuracy: 0.8784 - val_tumor_presence_auc: 0.9621 - val_tumor_presence_loss: 0.0731 - val_tumor_presence_precisi

143/143 ━━━━━━━━━━━━━━━━━━━━ 35s 241ms/step - loss: 0.5611 - tumor_presence_accuracy: 0.9421 - tumor_presence_auc: 0.9789 - tumor_presence_loss: 0.0429 - tumor_presence_precision: 0.9513 - tumor_presence_recall: 0.9695 - tumor_type_accuracy: 0.5995 - tumor_type_loss: 0.3986 - val_loss: 1.3495 - val_tumor_presence_accuracy: 0.8443 - val_tumor_presence_auc: 0.9083 - val_tumor_presence_loss: 0.0962 - val_tumor_presence_precision: 0.8939 - val_tumor_presence_recall: 0.8896 - val_tumor_type_accuracy: 0.4007 - val_tumor_type_loss: 0.9367 - learning_rate: 0.0010
Epoch 7/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 0.5447 - tumor_presence_accuracy: 0.9456 - tumor_presence_auc: 0.9783 - tumor_presence_loss: 0.0436 - tumor_presence_precision: 0.9579 - tumor_presence_recall: 0.9674 - tumor_type_accuracy: 0.6120 - tumor_type_loss: 0.3855

143/143 ━━━━━━━━━━━━━━━━━━━━ 37s 255ms/step - loss: 0.5448 - tumor_presence_accuracy: 0.9456 - tumor_presence_auc: 0.9784 - tumor_presence_loss: 0.0435 - tumor_presence_precision: 0.9579 - tumor_presence_recall: 0.9674 - tumor_type_accuracy: 0.6119 - tumor_type_loss: 0.3856 - val_loss: 0.8679 - val_tumor_presence_accuracy: 0.8871 - val_tumor_presence_auc: 0.9646 - val_tumor_presence_loss: 0.0680 - val_tumor_presence_precision: 0.9690 - val_tumor_presence_recall: 0.8714 - val_tumor_type_accuracy: 0.5389 - val_tumor_type_loss: 0.5973 - learning_rate: 0.0010
Epoch 8/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 175ms/step - loss: 0.5142 - tumor_presence_accuracy: 0.9426 - tumor_presence_auc: 0.9787 - tumor_presence_loss: 0.0447 - tumor_presence_precision: 0.9582 - tumor_presence_recall: 0.9629 - tumor_type_accuracy: 0.6197 - tumor_type_loss: 0.3611 - val_loss: 7.6670 - val_tumor_presence_accuracy: 0.3132 - val_tumor_presence_auc: 0.8798 - val_tumor_presence_loss: 1.5115 - val_tumor_presence_precisi

143/143 ━━━━━━━━━━━━━━━━━━━━ 36s 251ms/step - loss: 0.4293 - tumor_presence_accuracy: 0.9540 - tumor_presence_auc: 0.9872 - tumor_presence_loss: 0.0340 - tumor_presence_precision: 0.9679 - tumor_presence_recall: 0.9686 - tumor_type_accuracy: 0.6289 - tumor_type_loss: 0.3040 - val_loss: 0.5263 - val_tumor_presence_accuracy: 0.9493 - val_tumor_presence_auc: 0.9843 - val_tumor_presence_loss: 0.0373 - val_tumor_presence_precision: 0.9443 - val_tumor_presence_recall: 0.9879 - val_tumor_type_accuracy: 0.6010 - val_tumor_type_loss: 0.3664 - learning_rate: 5.0000e-04
Epoch 15/50
143/143 ━━━━━━━━━━━━━━━━━━━━ 25s 174ms/step - loss: 0.4477 - tumor_presence_accuracy: 0.9502 - tumor_presence_auc: 0.9890 - tumor_presence_loss: 0.0318 - tumor_presence_precision: 0.9670 - tumor_presence_recall: 0.9646 - tumor_type_accuracy: 0.6323 - tumor_type_loss: 0.3199 - val_loss: 1.1032 - val_tumor_presence_accuracy: 0.8784 - val_tumor_presence_auc: 0.9854 - val_tumor_presence_loss: 0.0635 - val_tumor_presence_pr

2026/01/28 16:15:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 16:16:12 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpqy9myhfw/model, flavor: tensorflow). Fall back to return ['tensorflow==2.19.0', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 
Registered model 'BrainTumorMRI_DenseNet121_2Head' already exists. Creating a new version of this model...
2026/01/28 16:16:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BrainTumorMRI_DenseNet121_2Head, version 15
Created version '15' of model 'BrainTumorMRI_DenseNet121_2Head'.
/tmp/ipykernel_55/1507103595.py:39: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecat

🏃 View run DenseNet121freeze=True_mask=True_20260128-1606 at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173/runs/ae881942e1c54e0c9ed85c8b954980fe
🧪 View experiment at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173


In [99]:
Warning : do not forget :
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning
- maximize reccal (y_pred > 0.3 → tumeur)

SyntaxError: invalid character '→' (U+2192) (1940605460.py, line 4)